In [3]:
import os
import re
import numpy as np
import imageio.v3 as iio
import json # Import the json library

# --- Configuration ---
# Base path where your main data and results folders are located
data_path = r"E:\20250404_DNA_PAINT_12 targets_ON testing"

# New subfolder to save ALL cropped comparison images
comparison_results_root_subfolder = "stitching_comparisons"
comparison_results_root_dir = os.path.join(data_path, comparison_results_root_subfolder)

# Name of the JSON coordinate file
coordinates_filename = "checkStitching_coordinates.json"
coordinates_filepath = os.path.join(os.getcwd(), coordinates_filename) # Assumes file is in current working directory

# --- Create root output directory for comparisons if it doesn't exist ---
os.makedirs(comparison_results_root_dir, exist_ok=True)
print(f"Ensured root output directory for comparisons exists: {comparison_results_root_dir}")

# --- 1. Read the cropping instructions from the JSON file ---
crop_instructions = []
try:
    with open(coordinates_filepath, 'r') as f:
        crop_instructions = json.load(f)
    print(f"\nSuccessfully read {len(crop_instructions)} cropping instruction(s) from '{coordinates_filename}'.")

except FileNotFoundError:
    print(f"ERROR: JSON coordinates file not found at '{coordinates_filepath}'. Please create it.")
    exit()
except json.JSONDecodeError as e:
    print(f"ERROR parsing JSON file '{coordinates_filepath}': {e}")
    exit()
except Exception as e:
    print(f"ERROR reading coordinates file '{coordinates_filepath}': {e}")
    exit()

# --- Discover all 'analysis_results_' directories ---
analysis_dirs = []
analysis_dir_pattern = re.compile(r"analysis_results_.*") # Matches analysis_results_*, analysis_results_maxIntAtOverlap, etc.

for item in os.listdir(data_path):
    full_path = os.path.join(data_path, item)
    if os.path.isdir(full_path) and analysis_dir_pattern.match(item):
        analysis_dirs.append(full_path)

if not analysis_dirs:
    print(f"No directories starting with 'analysis_results_' found in '{data_path}'. Exiting.")
    exit()
else:
    print(f"\nFound {len(analysis_dirs)} analysis result directories to process:")
    for adir in analysis_dirs:
        print(f"  - {os.path.basename(adir)}")

# --- Process each analysis results directory ---
for analysis_dir_idx, current_analysis_results_dir in enumerate(analysis_dirs):
    analysis_dir_name = os.path.basename(current_analysis_results_dir)
    
    # Extract the suffix from the analysis directory name
    # e.g., from "analysis_results_maxIntAtOverlap" get "_maxIntAtOverlap"
    suffix_match = re.match(r"analysis_results(.*)", analysis_dir_name)
    suffix_to_add = suffix_match.group(1) if suffix_match else "" # Will be "" if no suffix after analysis_results

    print(f"\n=====================================================================")
    print(f"Processing images from source directory: '{analysis_dir_name}' ({analysis_dir_idx + 1}/{len(analysis_dirs)})")
    print(f"  Adding suffix '{suffix_to_add}' to output filenames.")
    print(f"=====================================================================")

    # --- Process each cropping instruction for the current analysis directory ---
    if not crop_instructions:
        print("  No cropping instructions found in the JSON file. Skipping this source directory.")
        continue

    for idx, instruction in enumerate(crop_instructions):
        print(f"\n  --- Processing Cropping Instruction {idx + 1} ---")

        # Validate instruction parameters
        required_params = ['startsWith', 'x', 'y', 'width']
        if not all(param in instruction for param in required_params):
            print(f"  ERROR: Instruction {idx + 1} is missing one or more required parameters ({', '.join(required_params)}). Skipping.")
            continue

        starts_with_prefix = os.path.basename(instruction['startsWith']) # Ensure only filename prefix is used
        crop_x = instruction['x']
        crop_y = instruction['y']
        crop_width = instruction['width']
        crop_height = crop_width # Assuming square crop as per request

        print(f"    Target Prefix: '{starts_with_prefix}'")
        print(f"    Crop X: {crop_x}, Crop Y: {crop_y}, Crop Width: {crop_width}")

        # --- 2. Find the target image file within the CURRENT analysis directory ---
        target_image_filepath = None
        file_pattern = re.compile(re.escape(starts_with_prefix) + r".*\.tif$", re.IGNORECASE)

        for filename in os.listdir(current_analysis_results_dir):
            if file_pattern.match(filename):
                target_image_filepath = os.path.join(current_analysis_results_dir, filename)
                break

        if target_image_filepath is None:
            print(f"    ERROR: No TIFF file found in '{current_analysis_results_dir}' starting with '{starts_with_prefix}'. Skipping this instruction for this source.")
            continue

        print(f"    Found target image file: '{os.path.basename(target_image_filepath)}'")

        # --- 3. Load the image and perform cropping ---
        try:
            image = iio.imread(target_image_filepath)
            print(f"    Original image shape: {image.shape}")

            # Check if crop coordinates are within image bounds
            img_height, img_width = image.shape[:2] # Get height and width, ignoring channels if present

            if not (0 <= crop_x < img_width and 0 <= crop_y < img_height):
                print(f"    WARNING: Crop start coordinates ({crop_x}, {crop_y}) are outside image dimensions ({img_width}, {img_height}).")
                # Adjust crop_x, crop_y to be within bounds if they are negative or too large
                crop_x = max(0, min(crop_x, img_width - 1))
                crop_y = max(0, min(crop_y, img_height - 1))
                print(f"    Adjusted crop start to ({crop_x}, {crop_y}).")

            # Calculate actual crop end points, ensuring they don't exceed image boundaries
            crop_end_x = min(crop_x + crop_width, img_width)
            crop_end_y = min(crop_y + crop_height, img_height)

            # Adjust width/height if they were clipped
            actual_crop_width = crop_end_x - crop_x
            actual_crop_height = crop_end_y - crop_y

            if actual_crop_width <= 0 or actual_crop_height <= 0:
                print(f"    ERROR: Cropping resulted in zero or negative dimensions. Cannot crop image. Skipping.")
                continue

            cropped_image = image[crop_y:crop_end_y, crop_x:crop_end_x]
            print(f"    Cropped image shape: {cropped_image.shape}")

            # --- 4. Save the cropped image ---
            original_base_name = os.path.splitext(os.path.basename(target_image_filepath))[0]
            
            # NEW: Incorporate the suffix into the filename
            output_cropped_filename = f"{original_base_name}{suffix_to_add}_cropped_x{crop_x}_y{crop_y}_w{crop_width}.tif"
            output_cropped_filepath = os.path.join(comparison_results_root_dir, output_cropped_filename)

            iio.imwrite(output_cropped_filepath, cropped_image)
            print(f"    Successfully saved cropped image to: '{output_cropped_filepath}'")

        except Exception as e:
            print(f"    ERROR processing image '{os.path.basename(target_image_filepath)}' for this instruction: {e}")
            continue

print("\nAll analysis result directories and cropping instructions processed.")

Ensured root output directory for comparisons exists: E:\20250404_DNA_PAINT_12 targets_ON testing\stitching_comparisons

Successfully read 1 cropping instruction(s) from 'checkStitching_coordinates.json'.

Found 3 analysis result directories to process:
  - analysis_results_maxIntAtOverlap
  - analysis_results_avgIntAtOverlap
  - analysis_results_FusionStitcher

Processing images from source directory: 'analysis_results_maxIntAtOverlap' (1/3)
  Adding suffix '_maxIntAtOverlap' to output filenames.

  --- Processing Cropping Instruction 1 ---
    Target Prefix: 'Round8'
    Crop X: 5600, Crop Y: 1500, Crop Width: 1000
    Found target image file: 'Round8_Vimentin_A5_30_300_Stitched_MIP.tif'
    Original image shape: (8123, 8131)
    Cropped image shape: (1000, 1000)
    Successfully saved cropped image to: 'E:\20250404_DNA_PAINT_12 targets_ON testing\stitching_comparisons\Round8_Vimentin_A5_30_300_Stitched_MIP_maxIntAtOverlap_cropped_x5600_y1500_w1000.tif'

Processing images from source